# LLaMA Fine-tuning + MMLU Benchmark (Adaptive Learning)

**Part 1** fine-tune **LLaMA 7B** on **SciQ** (adaptive video learning: Evaluation + Recommendation). **Part 2** evaluate on **MMLU** (Accuracy, F1). Same workflow as the Qwen notebook.

**Model:** Llama-2-7b-hf (Hugging Face; gated — may require huggingface-cli login)  
**Data:** SciQ (allenai/sciq) → MMLU (cais/mmlu) for benchmark

## 1. Setup and Installation

In [ ]:
# Install required packages (same as Mistral notebooks)
!pip install -q datasets transformers accelerate peft bitsandbytes trl torch

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
import random

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load SciQ Dataset

In [ ]:
# Load SciQ dataset from HuggingFace (same as Mistral notebook)
dataset = load_dataset("allenai/sciq")
print("Dataset structure:", dataset)
print(f"\nTraining: {len(dataset['train'])}, Validation: {len(dataset['validation'])}, Test: {len(dataset['test'])}")
print("\nSample:", dataset['train'][0])

## 3. Prepare Dataset for Fine-tuning (same prompt format as Mistral)

In [ ]:
def create_training_prompt(example):
    """Correct answer scenario - same format as Mistral notebook."""
    context = example['support']
    question = example['question']
    correct_answer = example['correct_answer']
    return f"""### Context (Video Transcript):
{context}

### Assessment Question:
{question}

### Student Answer:
{correct_answer}

### Evaluation:
The student has demonstrated strong understanding of the concept. Their answer correctly identifies {correct_answer}, showing they've grasped the key principles from the lesson.

### Recommendation:
The student is ready to progress to more advanced topics building on this foundation. Recommend moving forward to the next concept in the learning sequence."""

def create_incorrect_prompt(example):
    """Incorrect answer scenario using distractors."""
    context = example['support']
    question = example['question']
    correct_answer = example['correct_answer']
    distractors = [example['distractor1'], example['distractor2'], example['distractor3']]
    incorrect_answer = random.choice([d for d in distractors if d])
    if not incorrect_answer:
        return None
    return f"""### Context (Video Transcript):
{context}

### Assessment Question:
{question}

### Student Answer:
{incorrect_answer}

### Evaluation:
The student's answer shows a misconception. They answered '{incorrect_answer}' but the correct answer is '{correct_answer}'. This indicates they need to review the foundational concepts.

### Recommendation:
The student should review prerequisite material before proceeding. Recommend watching an introductory video on this topic or revisiting related foundational concepts."""

# Build train/val datasets (correct + incorrect examples)
def prepare_dataset(dataset_split):
    prompts = []
    for example in dataset_split:
        prompts.append(create_training_prompt(example))
        inc = create_incorrect_prompt(example)
        if inc:
            prompts.append(inc)
    return prompts

train_prompts = prepare_dataset(dataset['train'])
val_prompts = prepare_dataset(dataset['validation'])
train_dataset = Dataset.from_dict({"text": train_prompts})
val_dataset = Dataset.from_dict({"text": val_prompts})
print(f"Train examples: {len(train_prompts)}, Val examples: {len(val_prompts)}")

## 4. Load LLaMA 7B Model with QLoRA

In [ ]:
# LLaMA 2 7B (meta-llama/Llama-2-7b-hf — gated; or use NousResearch/Llama-2-7b-hf)
MODEL_NAME = "meta-llama/Llama-2-7b-hf"  # Hugging Face login required for gated model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("LLaMA model and tokenizer loaded successfully!")

## 5. Configure LoRA for Efficient Fine-tuning

In [ ]:
# LoRA config - LLaMA uses q_proj, k_proj, v_proj, o_proj
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = prepare_model_for_kbit_training(model)
print("Model prepared for training!")

## 6. Training Configuration and SFTTrainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./adaptive-learning-llama",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=20,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args,
)
print("Trainer initialized and ready!")

## 7. Start Training

In [ ]:
print("Starting LLaMA fine-tuning on SciQ...")
trainer.train()
print("Training completed!")

## 8. Save the Fine-tuned LLaMA Model

In [ ]:
OUTPUT_DIR = "./adaptive-learning-llama-final"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model and tokenizer saved to {OUTPUT_DIR}")

---
# Part 2: MMLU Benchmark Evaluation

Evaluate the fine-tuned LLaMA model on **MMLU** (Massive Multitask Language Understanding). Same metrics as Mistral benchmark: **Accuracy**, **F1**.

## 9. Load MMLU Dataset

In [ ]:
# Load MMLU (same as Mistral benchmark notebook)
subjects = [
    "abstract_algebra", "anatomy", "college_mathematics", "elementary_mathematics",
    "high_school_biology", "high_school_chemistry", "high_school_physics",
]
mmlu_data = {}
for subject in subjects:
    try:
        ds = load_dataset("cais/mmlu", subject, split="test")
        mmlu_data[subject] = ds
        print(f"✓ {subject}: {len(ds)} questions")
    except Exception as e:
        print(f"✗ {subject}: {e}")
print(f"\nTotal subjects loaded: {len(mmlu_data)}")

## 10. Load Fine-tuned LLaMA Model (for evaluation)

**If you just ran Part 1**, you already have `model` and `tokenizer` in memory — skip the next cell and go to section 11.  
**If you are running Part 2 only** (e.g. new session), run the next cell to load from disk.

In [ ]:
# Load base LLaMA + PEFT adapter (run this only when starting Part 2 in a new session)
MODEL_PATH = "./adaptive-learning-llama-final"  # same as OUTPUT_DIR in Part 1

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model.eval()
print("Fine-tuned LLaMA model loaded for evaluation!")

## 11. Create MMLU Prompt and Evaluation Functions

In [ ]:
def create_mmlu_prompt(question, choices, student_choice_idx, correct_idx, subject):
    """Format MMLU question: student_choice_idx = which answer the student gave; correct_idx = true answer. Same as Mistral benchmark."""
    choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
    student_letter = chr(65 + student_choice_idx)
    domain = "MATH" if "math" in subject.lower() else ("SCIENCE" if any(x in subject.lower() for x in ["physics", "chemistry", "biology", "anatomy"]) else "REASONING")
    return f"""### Domain: {domain}
### Context:
Assessment on {subject.replace('_', ' ').title()}
### Assessment:
{question}
{choices_text}
### Student Answer: {student_letter}
### User Preference: videos
### Evaluation:"""

def parse_evaluation(response):
    """Extract correct/incorrect from model output (Evaluation section)."""
    if "### Evaluation:" in response:
        eval_part = response.split("### Evaluation:")[1]
        if "### Recommendation:" in eval_part:
            eval_part = eval_part.split("### Recommendation:")[0]
        eval_lower = eval_part.strip().lower()
        if "correct" in eval_lower and "incorrect" not in eval_lower[:50]:
            return 1
        if "misconception" in eval_lower or "incorrect" in eval_lower:
            return 0
    return 1  # default

## 12. Run MMLU Evaluation

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

samples_per_subject = 20
all_predictions = []
all_ground_truth = []

for subject, ds in mmlu_data.items():
    indices = random.sample(range(len(ds)), min(samples_per_subject, len(ds)))
    for idx in tqdm(indices, desc=subject):
        item = ds[idx]
        question = item["question"]
        choices = item["choices"]
        correct_idx = item["answer"]
        # Scenario 1: student gave correct answer -> ground truth 1
        prompt = create_mmlu_prompt(question, choices, correct_idx, correct_idx, subject)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        all_predictions.append(parse_evaluation(response))
        all_ground_truth.append(1)
        # Scenario 2: student gave wrong answer -> ground truth 0
        wrong_idx = (correct_idx + 1) % len(choices) if len(choices) > 1 else 0
        prompt = create_mmlu_prompt(question, choices, wrong_idx, correct_idx, subject)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        all_predictions.append(parse_evaluation(response))
        all_ground_truth.append(0)
print("Evaluation done.")

## 13. Overall Metrics (Accuracy, F1)

In [ ]:
accuracy = accuracy_score(all_ground_truth, all_predictions)
f1 = f1_score(all_ground_truth, all_predictions, average="macro")
print("=" * 80)
print("LLaMA Fine-tuned Model - MMLU Benchmark Results")
print("=" * 80)
print(f"Accuracy              {accuracy:.4f}")
print(f"F1 Score (macro)      {f1:.4f}")
print("=" * 80)
print("\nClassification Report:")
print(classification_report(all_ground_truth, all_predictions))